# Week 41, Day 5: Motor adaptation: learning at two speeds

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Learning curve and aftereffect

Write one_rate(f, A, B): starting from x = 0, record the output x on each trial, then update x with the error f[n] - x. Return the outputs. Apply it to 200 trials of the field (f = 1) followed by 50 trials without it (f = 0).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def one_rate(f, A, B):
    x, out = 0.0, []
    for fn in f:
        ...
    return np.array(out)

A1, B1, nA = 0.99, 0.1, 200
f = np.r_[np.ones(nA), np.zeros(50)]
x1 = one_rate(f, A1, B1)
print(f"output after 200 field trials {x1[nA - 1]:.3f} (steady state {B1 / (1 - A1 + B1):.3f}); first error without the field {0 - x1[nA]:.3f}")
plt.plot(x1, label="output"); plt.plot(f, "k--", lw=0.8, label="field"); plt.xlabel("trial"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_asymptote():
    "output approaches the steady state B/(1 - A + B) during the field"
    assert abs(x1[nA - 1] - B1 / (1 - A1 + B1)) < 0.02

def test_after():
    "after the field is switched off, the first error is opposite in sign to the first error in the field"
    assert (0 - x1[nA]) < 0 < (1 - x1[0])

test_asymptote()
test_after()
print("All checks passed")

### Exercise: Spontaneous recovery

Write two_rate_trial(xf, xs, e) returning the updated fast and slow states. The paradigm runner, which de-adapts until the output first reaches zero and then clamps errors, is given and runs both models.

In [ ]:
Af, Bf, As, Bs = 0.59, 0.21, 0.992, 0.02

def two_rate_trial(xf, xs, e):
    return ...

def paradigm(update, state, output, n_adapt=200, n_clamp=100):
    outs, phase, clamp_start = [], "adapt", None
    for n in range(n_adapt + 200 + n_clamp):
        x = output(state); outs.append(x)
        if phase == "adapt" and n == n_adapt: phase = "deadapt"
        if phase == "deadapt" and x <= 0: phase, clamp_start = "clamp", n
        f = 1.0 if phase == "adapt" else -1.0
        e = 0.0 if phase == "clamp" else f - x
        state = update(state, e)
        if clamp_start is not None and n >= clamp_start + n_clamp: break
    return np.array(outs), clamp_start

out2, clamp_start = paradigm(lambda s, e: two_rate_trial(*s, e), (0.0, 0.0), lambda s: s[0] + s[1])
out1, cs1 = paradigm(lambda s, e: A1 * s + B1 * e, 0.0, lambda s: s)
print(f"two-rate: de-adapted in {clamp_start - 200} trials, peak output during clamp {out2[clamp_start:].max():.2f}")
print(f"one-rate: de-adapted in {cs1 - 200} trials, peak output during clamp {out1[cs1:].max():.3f}")
plt.plot(out2, label="two-rate"); plt.plot(out1, label="single-rate"); plt.axhline(0, color="gray", lw=0.5)
plt.axvline(clamp_start, color="k", ls=":"); plt.xlabel("trial"); plt.ylabel("output"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_two_rate():
    "the two-rate model's output rebounds above 0.15 during the clamp"
    assert out2[clamp_start:].max() > 0.15

def test_one_rate():
    "the single-rate model shows no rebound (its clamp output stays below 0.02)"
    assert out1[cs1:].max() < 0.02

test_two_rate()
test_one_rate()
print("All checks passed")

### Exercise: Faster relearning

Run the two-rate model through 200 trials of the field, de-adaptation until the output reaches zero, then the field again. Count how many trials it takes to reach an output of 0.5 the first time (first_half) and after de-adaptation (relearn_half).

In [ ]:
xf = xs = 0.0; outs = []; phase = "learn1"; first_half = relearn_half = None; n = 0; t_relearn = None
while n < 600:
    x = xf + xs; outs.append(x)
    if phase == "learn1" and first_half is None and x >= 0.5: first_half = n
    if phase == "learn1" and n == 200: phase = "deadapt"
    if phase == "deadapt" and x <= 0: phase, t_relearn = "learn2", n
    if phase == "learn2" and relearn_half is None and x >= 0.5: relearn_half = ...
    f = -1.0 if phase == "deadapt" else 1.0
    xf, xs = two_rate_trial(xf, xs, f - x)
    n += 1
    if relearn_half is not None: break
print(f"trials to reach 0.5: first time {first_half}, relearning {relearn_half}")
plt.plot(outs); plt.axhline(0.5, color="gray", ls=":"); plt.xlabel("trial"); plt.ylabel("output"); plt.show()

In [ ]:
import numpy as np
def test_savings():
    "the two-rate model reaches half adaptation faster on relearning than at first"
    assert relearn_half < first_half

test_savings()
print("All checks passed")